# ML · Predicción — Riesgo de Retraso en Entrega

## Propósito

Aplicar el modelo `champion` registrado en Unity Catalog sobre los pedidos vigentes
y escribir las predicciones en una tabla Gold para consumo por dashboards y alertas.

## Fuentes

- `gld_dataco.supply_chain.desc_ml_features_orders` — features de pedidos vigentes
- `gld_dataco.supply_chain.ml_late_delivery_risk` — modelo registrado (alias `champion`)

## Destino

**Tabla:** `gld_dataco.supply_chain.desc_ml_late_delivery_risk`

**Grano:** un registro por pedido vigente

## Columnas de salida

| Columna | Tipo | Descripción |
|---|---|---|
| `order_id` | String | Identificador del pedido |
| `late_delivery_risk_pred` | Int | Predicción del modelo (0 o 1) |
| `prob_retraso` | Double | Probabilidad de retraso (clase 1) |
| `late_delivery_risk_original` | Int | Valor real (para comparación y monitoreo) |
| `model_version` | String | Versión del modelo `champion` usada |
| `fecha_prediccion` | Timestamp | Momento de la predicción |
| `fecha_fin_corte` | Date | Corte del pedido (trazabilidad) |

## Notas

- La tabla se recrea en cada ejecución (overwrite idempotente)
- `prob_retraso` permite construir alertas con umbral configurable (ej. > 0.7)
- El modelo se carga por alias `champion`, no por número de versión,
  para que al promover una nueva versión el notebook no requiera cambios

In [0]:
# ============================================================
# ML_PREDICT — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog",     "gld_dataco",    "Catálogo Gold")
dbutils.widgets.text("gold_schema",      "supply_chain",  "Esquema Gold")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema",  "supply_chain",  "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo si no corre dentro del Workflow)")

gold_catalog     = dbutils.widgets.get("gold_catalog")
gold_schema      = dbutils.widgets.get("gold_schema")
metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema  = dbutils.widgets.get("metadata_schema")

GOLD_ML_FEATURES_FQN   = f"{gold_catalog}.{gold_schema}.desc_ml_features_orders"
MODEL_NAME_UC          = f"{gold_catalog}.{gold_schema}.ml_late_delivery_risk"
GOLD_ML_PREDICT_FQN    = f"{gold_catalog}.{gold_schema}.desc_ml_late_delivery_risk"
TABLA_ETL_LOG          = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"

print(f"Tabla de features   : {GOLD_ML_FEATURES_FQN}")
print(f"Modelo UC champion  : {MODEL_NAME_UC}")
print(f"Tabla de predicciones: {GOLD_ML_PREDICT_FQN}")

In [0]:
# ============================================================
# ML_PREDICT — Celda 2: Inicio de log de ejecución
# ============================================================

import uuid

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID   = str(uuid.uuid4())
NOTEBOOK = "ml_predict"
CAPA     = "gold"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("Corrida manual: log_id no se publica como Task Value")

print(f"Registro EN_PROCESO creado: {LOG_ID}")

In [0]:
# ============================================================
# ML_PREDICT — Celda 3: Cargar modelo champion desde Unity Catalog
# ============================================================

import mlflow
import mlflow.spark
from mlflow import MlflowClient

# UC Volume como directorio temporal (requerido en clusters shared/serverless)
UC_TMP_DIR = "/Volumes/gld_dataco/supply_chain/ml_tmp"

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient(registry_uri="databricks-uc")

# Obtener la versión actual del alias champion
version_champion = client.get_model_version_by_alias(
    name=MODEL_NAME_UC,
    alias="champion"
)
numero_version = version_champion.version
model_uri      = f"models:/{MODEL_NAME_UC}@champion"

# Cargar el modelo
modelo = mlflow.spark.load_model(model_uri, dfs_tmpdir=UC_TMP_DIR)

print(f"Modelo cargado    : {MODEL_NAME_UC}")
print(f"Version champion  : {numero_version}")
print(f"Model URI         : {model_uri}")

In [0]:
# ============================================================
# ML_PREDICT — Celda 4: Cargar features y aplicar predicciones
# ============================================================

from pyspark.sql import functions as F

df_features = (
    spark.table(GOLD_ML_FEATURES_FQN)
    .select(
        "order_id",
        "shipping_mode",
        "days_for_shipment_scheduled",
        "market",
        "order_region",
        "es_sospechoso_fraude",
        "mes_pedido",
        "dia_semana_pedido",
        "late_delivery_risk",
        "fecha_fin_corte"
    )
    .dropna(subset=[
        "shipping_mode", "market", "order_region",
        "days_for_shipment_scheduled", "es_sospechoso_fraude",
        "mes_pedido", "dia_semana_pedido"
    ])
)

# Aplicar el pipeline del modelo
df_pred_raw = modelo.transform(df_features)

# Extraer probabilidad de la clase 1 (retraso)
extraer_prob = F.udf(lambda v: float(v[1]))

df_predicciones = (
    df_pred_raw
    .select(
        F.col("order_id"),
        F.col("prediction").cast("int").alias("late_delivery_risk_pred"),
        extraer_prob(F.col("probability")).cast("double").alias("prob_retraso"),
        F.col("late_delivery_risk").alias("late_delivery_risk_original"),
        F.lit(numero_version).alias("model_version"),
        F.current_timestamp().alias("fecha_prediccion"),
        F.col("fecha_fin_corte")
    )
)

total_pred     = df_predicciones.count()
pred_retraso   = df_predicciones.filter(F.col("late_delivery_risk_pred") == 1).count()
pred_a_tiempo  = total_pred - pred_retraso

print(f"Total de predicciones       : {total_pred}")
print(f"Prediccion retraso (1)      : {pred_retraso} ({round(pred_retraso/total_pred*100,1)}%)")
print(f"Prediccion a tiempo (0)     : {pred_a_tiempo} ({round(pred_a_tiempo/total_pred*100,1)}%)")

In [0]:
# ============================================================
# ML_PREDICT — Celda 5: Guardar tabla de predicciones
# ============================================================

(
    df_predicciones
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ML_PREDICT_FQN)
)

filas_escritas = spark.table(GOLD_ML_PREDICT_FQN).count()

print(f"Tabla de predicciones guardada : {GOLD_ML_PREDICT_FQN}")
print(f"Registros escritos             : {filas_escritas}")

In [0]:
# ============================================================
# ML_PREDICT — Celda 6: Cerrar log de ejecución
# ============================================================

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado           = 'EXITOSO',
    fecha_fin        = current_timestamp(),
    filas_procesadas = {filas_escritas}
WHERE log_id = '{LOG_ID}'
""")

print(f"etl_log actualizado: EXITOSO ({filas_escritas} predicciones en {GOLD_ML_PREDICT_FQN})")
print(f"Modelo champion utilizado: {MODEL_NAME_UC} v{numero_version}")